# 02.00 · Ejecutar el flujo completo

Se ejecutan en orden los notebooks `02.01` a `02.05`. El orden importa: cada uno lee el CSV
que deja el anterior en `data/interim/`. Cada notebook corre en su propio kernel, con el Python
de `.venv` y el directorio de trabajo `notebooks/2. IDOs - Input CCI/`.

Entre el `02.04` y el `02.05` se **aprueba el concepto de priorización** de los IDOs nuevos:
por cada uno se muestra lo que redactó Azure y se pregunta si se aprueba o se edita. Lo aprobado
es lo que llega al Excel del `02.05`.

Los notebooks originales no se modifican; sí se actualizan sus archivos de datos. El flujo se
detiene en el primer fallo. Al final se muestra un resumen y se agrega al histórico
`reports/2. IDOs - Input CCI_historico_ejecuciones.xlsx`, que guarda una fila por notebook en cada corrida.

**Antes de ejecutar, se cierran en Positron los notebooks del flujo.** `02.02`, `02.03` y
`02.04` llaman a Azure y necesitan conexión; los IDOs que ya tienen respuesta guardada no
vuelven a llamar. Si faltan nbclient o nbformat, se ejecuta `uv sync` en la raíz.

### Parámetros

`notebooks_a_ejecutar` lleva los notebooks en el orden del flujo; se pueden quitar algunos para
correr solo una parte, conservando el orden. `timeout_por_celda` está en segundos: 3.600 permite
hasta una hora por celda, porque una celda puede contener todas las llamadas a Azure.

In [12]:
notebooks_a_ejecutar = [
    "02.01_extraccion_texto_idos.ipynb",
    "02.02_redaccion_textos_idos.ipynb",
    "02.03_evidencia_repositorio.ipynb",
    "02.04_concepto_priorizacion.ipynb",
    "02.05_excel_input_cci.ipynb",
]
timeout_por_celda = 3600

## 1. LOAD LIBRARIES

nbformat lee los notebooks; nbclient ejecuta sus celdas en un kernel separado, y jupyter_client
permite fijar el Python del proyecto. Las rutas se toman de
`caf_input_cci_preparatoria/config.py`.

In [13]:
from datetime import datetime
import json
from pathlib import Path
import re
import subprocess
import sys
import tempfile
import textwrap
from time import perf_counter

from dotenv import dotenv_values
from IPython.core.error import StdinNotImplementedError
from IPython.display import display
from jupyter_client import KernelManager
from jupyter_client.kernelspec import KernelSpecManager
from nbclient import NotebookClient
import nbformat
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

Los valores del `.env` se leen sin imprimirlos, solo para reemplazarlos por `[OCULTO]` si
llegaran a aparecer en un mensaje de error.

El archivo `kernel.json` le indica a Jupyter que use `.venv/bin/python`. Se crea en una carpeta
temporal, sin registrar un kernel global, y esa carpeta se borra al terminar.

`idos_concepto.csv` es la salida del `02.04` y `conceptos_aprobados.csv`, el registro de
conceptos aprobados; los dos se usan en la aprobación, antes del `02.05`.

In [14]:
directorio_notebooks = ROOT / "notebooks" / "2. IDOs - Input CCI"
python_proyecto = ROOT / ".venv" / "bin" / "python"
assert python_proyecto.is_file(), "No se encuentra el intérprete del proyecto en .venv/bin/python"

archivo_historico = config.REPORTS / "2. IDOs - Input CCI_historico_ejecuciones.xlsx"
archivo_idos = config.INTERIM / "2. IDOs - Input CCI" / "02.01_extraccion_idos" / "idos.csv"
carpeta_concepto = config.INTERIM / "2. IDOs - Input CCI" / "02.04_concepto_priorizacion"
archivo_concepto = carpeta_concepto / "idos_concepto.csv"
archivo_registro = carpeta_concepto / "conceptos_aprobados.csv"
valores_privados = [valor for valor in dotenv_values(ROOT / ".env").values() if valor]

In [15]:
nombre_kernel = "caf-proyecto"
carpeta_temporal = tempfile.TemporaryDirectory()
directorio_kernel = Path(carpeta_temporal.name) / nombre_kernel
directorio_kernel.mkdir()
especificacion_kernel = {
    "argv": [str(python_proyecto), "-m", "ipykernel_launcher", "-f", "{connection_file}"],
    "display_name": "Python del proyecto CAF",
    "language": "python",
}
(directorio_kernel / "kernel.json").write_text(json.dumps(especificacion_kernel), encoding="utf-8")
gestor_especificaciones = KernelSpecManager(
    kernel_dirs=[carpeta_temporal.name], ensure_native_kernel=False
)

## 3. PROCESS DATA

Cada notebook se lee, se le borran las salidas anteriores y se ejecuta en un kernel nuevo, para
que no comparta variables con los demás. De la salida de `02.02`, `02.03` y `02.04` se toma la
línea `Uso de Azure: …`, con las llamadas y los tokens de entrada y de salida; en `02.01` y
`02.05`, que no usan Azure, esas columnas quedan vacías.

En la fila de `02.01` se anotan los IDOs leídos y los nuevos: los que tienen una huella que no
estaba en `idos.csv` antes de la corrida. Un Word modificado cuenta como nuevo, aunque conserve
el nombre, porque su contenido cambió. Si falla, se anota la celda del error (contando
también las de texto) y no se ejecutan los notebooks siguientes. El detalle completo del error
se ve abriendo y ejecutando ese notebook.

Primero se ejecutan los notebooks hasta el `02.04`; el `02.05` espera a la aprobación de los
conceptos.

In [16]:
def ocultar(texto):
    for valor in valores_privados:
        texto = texto.replace(valor, "[OCULTO]")
    return texto


def uso_de_azure(notebook):
    salidas = [
        salida.get("text", "") for celda in notebook.cells for salida in celda.get("outputs", [])
    ]
    patron = r"Uso de Azure: (\d+) llamadas; (\d+) tokens de entrada; (\d+) tokens de salida"
    encontrado = re.search(patron, "".join(salidas))
    if encontrado is None:
        return {}
    return dict(
        zip(["Llamadas Azure", "Tokens entrada", "Tokens salida"], map(int, encontrado.groups()))
    )


def leer_idos():
    if not archivo_idos.exists():
        return pd.DataFrame(columns=["archivo", "huella"])
    return pd.read_csv(
        archivo_idos, encoding="utf-8-sig", usecols=["archivo", "huella"], dtype=str
    )


def celda_con_error(notebook):
    for numero, celda in enumerate(notebook.cells, start=1):
        if any(salida.output_type == "error" for salida in celda.get("outputs", [])):
            return numero
    return None

In [17]:
def ejecutar(nombre):
    """Ejecuta un notebook, agrega su fila a `resultados` y devuelve True si falló."""
    print(f"Ejecutando {nombre}...")
    notebook = nbformat.read(directorio_notebooks / nombre, as_version=4)
    for celda in notebook.cells:
        if celda.cell_type == "code":
            celda.outputs, celda.execution_count = [], None
    cliente = NotebookClient(
        notebook,
        km=KernelManager(kernel_name=nombre_kernel, kernel_spec_manager=gestor_especificaciones),
        timeout=timeout_por_celda,
        resources={"metadata": {"path": str(directorio_notebooks)}},
    )
    inicio = perf_counter()
    error = ""
    try:
        cliente.execute()
    except Exception as fallo:  # noqa: BLE001 - el fallo queda en el resumen y detiene el flujo
        tipo = getattr(fallo, "ename", type(fallo).__name__)
        error = ocultar(f"{tipo}: {getattr(fallo, 'evalue', fallo)}")
    resultados.append(
        {
            "Fecha y hora": fecha_ejecucion,
            "Notebook": nombre,
            "Estado": "Falló" if error else "Completado",
            "Duración (s)": round(perf_counter() - inicio, 1),
            **uso_de_azure(notebook),
            "Celda": celda_con_error(notebook) if error else None,
            "Error": error,
        }
    )
    if nombre.startswith("02.01") and not error:
        idos_despues = leer_idos()
        nuevos = idos_despues.loc[~idos_despues["huella"].isin(idos_antes["huella"]), "archivo"]
        resultados[-1].update(
            {
                "IDOs leídos": len(idos_despues),
                "IDOs nuevos": len(nuevos),
                "Archivos nuevos": "; ".join(nuevos),
            }
        )
    if error:
        print(f"Falló {nombre}, celda {resultados[-1]['Celda']}: {error}")
    return bool(error)


fecha_ejecucion = datetime.now().strftime("%Y-%m-%d %H:%M")
idos_antes = leer_idos()
resultados = []
antes_de_aprobar = [nombre for nombre in notebooks_a_ejecutar if not nombre.startswith("02.05")]
despues_de_aprobar = [nombre for nombre in notebooks_a_ejecutar if nombre.startswith("02.05")]
detenido = False
for nombre in antes_de_aprobar:
    detenido = ejecutar(nombre)
    if detenido:
        break

Ejecutando 02.01_extraccion_texto_idos.ipynb...
Ejecutando 02.02_redaccion_textos_idos.ipynb...
Ejecutando 02.03_evidencia_repositorio.ipynb...
Ejecutando 02.04_concepto_priorizacion.ipynb...


### Aprobación del concepto de priorización

Se revisan los IDOs con `estado_concepto` igual a `Pendiente` en `idos_concepto.csv`: los nuevos
y los Word modificados (cambió su huella). Por cada uno se muestra completo lo que redactó Azure
y se responde en la casilla de texto de Positron:

- **A**: se aprueban la decisión y el concepto de Azure tal cual (estado `Aprobado`).
- **E**: se crea `concepto_por_editar.txt` en la carpeta del `02.04` con la decisión y el
  concepto de Azure, y se abre en TextEdit. Se ajustan ahí, se guarda el archivo y se escribe
  **OK** en la casilla. La primera línea debe decir `Decisión final (Sí/No): Sí` o `No`; el
  concepto va después de la línea `Concepto final:` y se guarda en un solo párrafo. Si no se
  cambió nada, queda `Aprobado`; si cambió, `Editado`.

La casilla de Positron se cierra con una respuesta vacía cuando se hace clic fuera de ella (por
ejemplo, para pasar a TextEdit). Una respuesta vacía no hace nada: la pregunta vuelve a aparecer.

Cada respuesta se guarda de inmediato en dos lugares: se agrega una fila a
`conceptos_aprobados.csv`, el registro que ningún notebook reescribe, y se escriben
`decision_final`, `concepto_final` y `estado_concepto` en la fila del IDO en
`idos_concepto.csv`, que es lo que leen el `02.05` y el paso 3. Si se interrumpe la celda, lo ya
aprobado queda guardado y la próxima corrida solo pregunta por lo que falte. El registro
lo crea vacío el `02.04` la primera vez; con el registro vacío se pregunta por todos los IDOs
(con **A** se aprueba el concepto de Azure tal cual). Las filas `Previo` son de IDOs registrados
antes de existir este paso y no se preguntan.

Cuando el `02.00` corre sin ventana (por ejemplo, desde el `03.00`) no se puede preguntar: los
conceptos quedan `Pendiente` y el `02.05` lo marca en `Revisar`.

In [18]:
archivo_edicion = carpeta_concepto / "concepto_por_editar.txt"


def pedir(pregunta, opciones):
    """Repite la pregunta hasta recibir una opción, sin distinguir mayúsculas ni tilde.

    Una respuesta vacía (la casilla se cerró al hacer clic fuera) solo vuelve a preguntar.
    """
    validas = {opcion.casefold().replace("í", "i"): opcion for opcion in opciones}
    while True:
        respuesta = input(pregunta).strip().casefold().replace("í", "i")
        if respuesta in validas:
            return validas[respuesta]
        if respuesta:
            print(f"Responde una de estas opciones: {', '.join(opciones)}")


def imprimir(etiqueta, texto):
    """Imprime el texto partido en líneas de 100 caracteres para leerlo completo."""
    print(f"{etiqueta}:\n{textwrap.fill(texto, width=100)}\n")


def mostrar_borrador(ido):
    imprimir("Operación", f"{ido['operacion']} · {ido['pais']} · {ido['producto_agrupado']}")
    print(f"Archivo: {ido['archivo']}\n")
    print(f"Decisión de Azure: {ido['decision_sugerida']}\n")
    imprimir("Justificación", ido["justificacion_decision"])
    print(f"Controles: {ido['controles'] or 'sin alertas'}\n")
    imprimir(f"Concepto de Azure ({len(ido['concepto'])} caracteres)", ido["concepto"])


def leer_edicion():
    """Devuelve la decisión (Sí/No o None) y el concepto en un párrafo del archivo editado."""
    lineas = archivo_edicion.read_text(encoding="utf-8").splitlines()
    encabezado = re.fullmatch(
        r"\s*Decisión final \(Sí/No\):\s*(\S*)\s*", lineas[0] if lineas else ""
    )
    decision = None
    if encabezado:
        decision = {"si": "Sí", "no": "No"}.get(encabezado[1].casefold().replace("í", "i"))
    marca = next(
        (numero for numero, linea in enumerate(lineas) if linea.strip() == "Concepto final:"), None
    )
    concepto = (
        ""
        if marca is None
        else " ".join(linea.strip() for linea in lineas[marca + 1 :] if linea.strip())
    )
    return decision, concepto


def editar(ido):
    """Abre la decisión y el concepto de Azure en TextEdit y lee la versión guardada."""
    archivo_edicion.write_text(
        f"Decisión final (Sí/No): {ido['decision_sugerida']}\nConcepto final:\n{ido['concepto']}\n",
        encoding="utf-8",
    )
    subprocess.run(["open", "-t", str(archivo_edicion)], check=False)
    print(f"Se abrió {archivo_edicion.name} en TextEdit: ajusta, guarda y escribe OK aquí.")
    while True:
        pedir("Escribe OK cuando hayas guardado el archivo: ", ["OK"])
        decision, concepto = leer_edicion()
        if decision and concepto:
            imprimir(f"Decisión final: {decision}. Concepto final", concepto)
            return decision, concepto
        print(
            "Revisa el archivo: la primera línea debe ser 'Decisión final (Sí/No): Sí' o 'No' y "
            "el concepto debe ir después de la línea 'Concepto final:'. Guarda y escribe OK."
        )


def leer_csv(ruta):
    return pd.read_csv(ruta, encoding="utf-8-sig", dtype=str, keep_default_na=False)


def guardar_aprobacion(ido, decision, concepto, estado):
    """Agrega la fila al registro y escribe lo final en la fila del IDO en idos_concepto.csv."""
    fila = {
        "archivo": ido["archivo"],
        "huella": ido["huella"],
        "decision_azure": ido["decision_sugerida"],
        "concepto_azure": ido["concepto"],
        "decision_final": decision,
        "concepto_final": concepto,
        "estado": estado,
        "fecha_aprobacion": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    anteriores = [leer_csv(archivo_registro)] if archivo_registro.exists() else []
    registro = pd.concat([*anteriores, pd.DataFrame([fila])], ignore_index=True)
    registro.to_csv(archivo_registro, index=False, encoding="utf-8-sig")

    tabla = leer_csv(archivo_concepto)
    es_ido = tabla["archivo"].eq(ido["archivo"]) & tabla["huella"].eq(ido["huella"])
    finales = ["decision_final", "concepto_final", "estado_concepto", "borrador_azure_nuevo"]
    tabla.loc[es_ido, finales] = [decision, concepto, estado, ""]
    tabla.to_csv(archivo_concepto, index=False, encoding="utf-8-sig")


def aprobar(ido):
    """Pregunta por un IDO y agrega la respuesta al registro."""
    mostrar_borrador(ido)
    if pedir("[A] Apruebo / [E] Quiero editar: ", ["A", "E"]) == "A":
        decision, concepto, estado = ido["decision_sugerida"], ido["concepto"], "Aprobado"
    else:
        decision, concepto = editar(ido)
        sin_cambios = (decision, concepto) == (ido["decision_sugerida"], ido["concepto"])
        estado = "Aprobado" if sin_cambios else "Editado"
    guardar_aprobacion(ido, decision, concepto, estado)
    print(f"Guardado como {estado}.")

In [19]:
if not detenido:
    idos_concepto = leer_csv(archivo_concepto)
    pendientes = idos_concepto[idos_concepto["estado_concepto"].eq("Pendiente")]
    print(f"{len(pendientes)} conceptos por aprobar.")
    try:
        for numero, ido in enumerate(pendientes.to_dict("records"), start=1):
            print(f"\n===== Concepto {numero} de {len(pendientes)}")
            aprobar(ido)
    except StdinNotImplementedError:
        print(
            "Aprobación omitida: el notebook corre sin ventana; los conceptos quedan pendientes."
        )
    display(leer_csv(archivo_concepto)["estado_concepto"].value_counts())

2 conceptos por aprobar.

===== Concepto 1 de 2
Operación:
Proyecto “Transformación y Modernización del Instituto Nacional de Formación Profesional y
Capacitación para el Desarrollo Humano (INADEH) para el fortalecimiento de la Empleabilidad y el
Desarrollo del Talento Humano en Panamá” · Panamá · PPI

Archivo: CCI-IDO-Consolidación Plan INADEH v2.docx

Decisión de Azure: No

Justificación:
Aunque existe valor de aprendizaje por la correspondencia limitada de la evidencia escolar con los
USD 61,3 millones destinados a infraestructura y equipamiento del INADEH, no se cumple la
evaluabilidad porque el IDO no describe fases, criterios de elegibilidad ni zonas sin intervención
que sustenten una comparación.

Controles: sin alertas

Concepto de Azure (1010 caracteres):
La operación es estratégica para CAF y se alinea con las agendas de bienestar social inclusivo,
habilidades del siglo XXI, fortalecimiento institucional e inclusión, diversidad y género, al apoyar
la formación para el empleo 

estado_concepto
Aprobado    15
Editado      1
Name: count, dtype: int64

### Excel del Input CCI

Con los conceptos aprobados se ejecuta el `02.05`, si está en `notebooks_a_ejecutar`.

In [20]:
if not detenido:
    for nombre in despues_de_aprobar:
        detenido = ejecutar(nombre)
carpeta_temporal.cleanup()

Ejecutando 02.05_excel_input_cci.ipynb...


## 4. SAVE DATA

Se muestra el resumen de esta corrida, con una fila por notebook ejecutado, y el total de
llamadas y tokens de Azure; los notebooks que no llegaron a ejecutarse no aparecen. Si un
notebook de Azure falla antes de terminar, sus llamadas de esa corrida no quedan contadas. Las mismas filas se agregan al final de
`reports/2. IDOs - Input CCI_historico_ejecuciones.xlsx`, que se crea la primera vez.

In [21]:
columnas = [
    "Fecha y hora",
    "Notebook",
    "Estado",
    "Duración (s)",
    "IDOs leídos",
    "IDOs nuevos",
    "Archivos nuevos",
    "Llamadas Azure",
    "Tokens entrada",
    "Tokens salida",
    "Error",
    "Celda",
]
columnas_enteras = [
    "IDOs leídos",
    "IDOs nuevos",
    "Llamadas Azure",
    "Tokens entrada",
    "Tokens salida",
    "Celda",
]

resumen = pd.DataFrame(resultados).reindex(columns=columnas)
resumen[columnas_enteras] = resumen[columnas_enteras].astype("Int64")  # enteros, no 3.0
with pd.option_context("display.max_colwidth", None):
    display(resumen)
totales = resumen[["Llamadas Azure", "Tokens entrada", "Tokens salida"]].sum()
print(
    f"Azure en esta corrida: {totales['Llamadas Azure']} llamadas; "
    f"{totales['Tokens entrada']:,} tokens de entrada; "
    f"{totales['Tokens salida']:,} tokens de salida"
)

,Fecha y hora,Notebook,Estado,Duración (s),IDOs leídos,IDOs nuevos,Archivos nuevos,Llamadas Azure,Tokens entrada,Tokens salida,Error,Celda
0,2026-11-13 15:25,02.01_extraccion_texto_idos.ipynb,Completado,1.3,16,2,CCI-IDO-Consolidación Plan INADEH v2.docx; FR-228 guanay actualizado.docx,<NA>,<NA>,<NA>,,<NA>
1,2026-11-13 15:25,02.02_redaccion_textos_idos.ipynb,Completado,48.5,<NA>,<NA>,NaN,2,15091,2853,,<NA>
2,2026-11-13 15:25,02.03_evidencia_repositorio.ipynb,Completado,52.6,<NA>,<NA>,NaN,2,17103,2562,,<NA>
3,2026-11-13 15:25,02.04_concepto_priorizacion.ipynb,Completado,24.1,<NA>,<NA>,NaN,2,7102,1082,,<NA>
4,2026-11-13 15:25,02.05_excel_input_cci.ipynb,Completado,1.2,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,,<NA>


Azure en esta corrida: 6 llamadas; 39,296 tokens de entrada; 6,497 tokens de salida


In [22]:
anteriores = pd.read_excel(archivo_historico) if archivo_historico.exists() else None
historico = pd.concat([anteriores, resumen], ignore_index=True).reindex(columns=columnas)
historico[columnas_enteras] = historico[columnas_enteras].astype("Int64")
archivo_historico.parent.mkdir(parents=True, exist_ok=True)
historico.to_excel(archivo_historico, index=False)
print(f"{len(resumen)} filas agregadas a {archivo_historico.relative_to(ROOT)}")

5 filas agregadas a reports/2. IDOs - Input CCI_historico_ejecuciones.xlsx
